In [2]:
import numpy as np

Doing matrix operations over integers, rather than a field.

In [ ]:
def bezout(n: int, m: int) -> list[int]:
    ''' Given two integers n and m, find integers x, y such that
        x*n + y*m = gcd(n, m)
        Returns [x, y, gcd(n, m)] '''
    
    # Initialize: [x, y, value]
    x0, y0, r0 = 1, 0, n
    x1, y1, r1 = 0, 1, m
    
    while r1 != 0: # do euclidean algorithm
        q = r0 // r1
        x0, x1 = x1, x0 - q * x1
        y0, y1 = y1, y0 - q * y1
        r0, r1 = r1, r0 - q * r1
    
    return [x0, y0, r0]

In [92]:
def smith(matrix, order_divisors=False):
    '''Given a matrix M with integer coefficients, returns a triple:
    [L, D, R] of matrices with integer coefficients, where L and R are 
    invertible over Z, D is diagonal and LDR = M. If order_divisors, 
    then D is arranged so that a_j divides a_{j+1} along the diagonal.'''

    def row_reduce(matrix, pivots=False):
        ''' subroutine which does just the row operations. Returns
        [D, R] where R is invertible, D is reduced and M = DR'''
        M = np.array(matrix, dtype='i')
        n, m = M.shape
        L = np.identity(n, dtype='i') 


        # We will do row reduction, except multiplying row by const isn't 
        # allowed unles const = +- 1
        # will use euclidean algorithm to find invertible row operations
        def bezout(n: int, m: int) -> list[int]:
            ''' Given two integers n and m, find integers x, y such that
                x*n + y*m = gcd(n, m)
                Returns [x, y, gcd(n, m)] '''
            
            # Initialize: [x, y, value]
            x0, y0, r0 = 1, 0, n
            x1, y1, r1 = 0, 1, m
            
            while r1 != 0: # do euclidean algorithm
                q = r0 // r1
                x0, x1 = x1, x0 - q * x1
                y0, y1 = y1, y0 - q * y1
                r0, r1 = r1, r0 - q * r1
            
            return [x0, y0, r0]
        
        # Do row reduction and keep track of row operations with matrix L
        row = 0
        pivot_cols = []
        for col in range(m):
            # Find pivot            
            pivot_row = None
            for oth in range(row, n):
                if M[oth, col]:
                    pivot_row = oth
                    break
            if pivot_row is None:
                continue

            # Swap pivot row into position
            if pivot_row != row:
                M[[row, pivot_row]] = M[[pivot_row, row]]

                # Keep track with L matrix
                row_swap = np.identity(n, dtype='i')
                row_swap[pivot_row, pivot_row] = row_swap[row, row] = 0
                row_swap[pivot_row, row] = row_swap[row, pivot_row] = 1
                L @= row_swap # update L matrix

            # make the pivot positive
            if M[row, col] < 0:
                M[row] *= -1
                negate_row = np.identity(n, dtype='i')
                negate_row[row, row] = -1
                L @= negate_row

            # Eliminate other rows `oth` using bezout coefficients
            for oth in range(n):
                # if another row isn't a multiple of the pivot
                if M[oth, col] % M[row, col] != 0:
                    # perform invertible row operation to pass to gcd
                    piv, other = M[row, col], M[oth, col]
                    x, y, gcd = bezout(piv, other) # bezout coefficients
                    a, b = piv // gcd, other // gcd
                    M[row], M[oth] = x*M[row] + y*M[oth], a*M[oth] - b*M[row]
                    # now M[row, col] = gcd and M[oth, col] = 0

                    # encode this operation in the L matrix
                    bez_op = np.identity(n, dtype='i')
                    bez_op[row, row], bez_op[row, oth] = a, -y
                    bez_op[oth, row], bez_op[oth, oth] = b, x
                    # bez_op is determinant 1, so invertible over Z
                    L @= bez_op

            # now that we've made the row entries divisible by pivot,
            # we can finally eliminate by subtracting off pivot row
            for oth in range(n):
                if oth != row and M[oth, col] != 0:
                    # should be divisible by M[row, col] if above worked
                    q = M[oth, col] // M[row, col]
                    M[oth] -= q * M[row]

                    # encode in the R matrix
                    subtract_row = np.identity(n, dtype='i')
                    subtract_row[oth, row] = q
                    L @= subtract_row

            pivot_cols.append(col)
            row += 1
            if row == n:
                break
        return [L, M] if not pivots else [L, M, pivot_cols]

    L, M = row_reduce(matrix)
    # now do the columns by taking transpose
    col_reduce = row_reduce(M.T, pivots=True)
    R, D, pivots = col_reduce[0].T, col_reduce[1].T, col_reduce[2]
    return [L, D, R]



In [77]:
A = np.array([[4, 3, 2],[6, 5, 4], [10, 9, 8]])
B = np.array([[1, 1],[1, 1]])

In [127]:
A = np.random.randint(-2, 3, size=(4, 3))
#A = np.array([[4, 3, 2],[6, 5, 4], [10, 9, 8]])
L, D, R = smith(A)
A, L@D@R, L, D, R

(array([[ 0, -2,  1],
        [-2, -1,  2],
        [ 0,  2, -1],
        [ 2,  1, -1]]),
 array([[ 0, -2,  1],
        [-2, -1,  2],
        [ 0,  2, -1],
        [ 2,  1, -1]]),
 array([[ 1, -2, -1,  0],
        [ 0, -1, -2,  0],
        [ 0,  0,  1,  0],
        [ 0,  1,  1,  1]], dtype=int32),
 array([[-4,  0, -2],
        [-2,  0, -1],
        [ 0, -1,  0],
        [ 0,  0,  2]]),
 array([[-1,  1,  0],
        [ 0, -2,  1],
        [ 0,  1,  0]], dtype=int32))

In [117]:
B = A.copy();B
smith(B)

[array([[ 1,  0,  2,  0],
        [-1, -1,  2,  0],
        [ 2,  1, -1,  0],
        [-2, -1, -2,  1]], dtype=int32),
 array([[1, 0, 0],
        [0, 2, 0],
        [0, 0, 1],
        [0, 0, 0]]),
 array([[1, 0, 0],
        [0, 1, 0],
        [0, 0, 1]], dtype=int32)]